# Решения: Удалённая работа в крипто/Web3 (проект 22)

Ноутбук самодостаточен: заново создаёт датасет проекта 22 и решает все пять
упражнений. Значения-ориентиры — срез 2025 года (web3.career, CryptoJobsList,
Electric Capital Developer Report, Stack Overflow Survey).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.width', 180); pd.set_option('display.max_columns', 30)
np.random.seed(42)

# role -> (jr, mid, sr, openings_pct, token_pct, stack)
SNAPSHOT = {
    'Smart Contract Engineer': (90_000, 150_000, 220_000, 22, 0.25, 'Solidity/EVM'),
    'ZK Engineer':             (120_000, 190_000, 280_000, 3,  0.35, 'Circom/Rust/Math'),
    'Security Auditor':        (100_000, 170_000, 260_000, 6,  0.20, 'Solidity/Foundry'),
    'Protocol Engineer':       (110_000, 170_000, 250_000, 5,  0.35, 'Rust/Go'),
    'Rust Engineer':           (100_000, 160_000, 230_000, 9,  0.25, 'Rust (Solana/Cosmos)'),
    'Blockchain Backend':      (85_000,  140_000, 200_000, 13, 0.15, 'Go/Node/Postgres'),
    'DevOps / Infra':          (90_000,  145_000, 205_000, 8,  0.12, 'K8s/nodes/validators'),
    'Full-Stack Web3':         (80_000,  130_000, 190_000, 14, 0.15, 'TS/React/ethers.js'),
    'Frontend Web3':           (75_000,  120_000, 175_000, 16, 0.12, 'React/wagmi/viem'),
    'On-chain Data Analyst':   (80_000,  130_000, 180_000, 4,  0.12, 'SQL/Dune/Python'),
}
df = pd.DataFrame([(r, *v) for r, v in SNAPSHOT.items()],
    columns=['role','jr','mid','sr','openings_pct','token_pct','stack']).set_index('role')
print('Готово, ролей:', len(df))

## Упражнение 1. Стек-премия: Solidity / Rust / ZK

Сравниваем senior-зарплаты по ключевым стекам и считаем премию ZK над Solidity.

In [ ]:
stack_map = {
    'Solidity': ['Smart Contract Engineer', 'Security Auditor'],
    'Rust':     ['Rust Engineer', 'Protocol Engineer'],
    'ZK':       ['ZK Engineer'],
    'Go/Node':  ['Blockchain Backend', 'DevOps / Infra'],
    'TS/React': ['Full-Stack Web3', 'Frontend Web3'],
    'SQL/Data': ['On-chain Data Analyst'],
}
stack_sr = {k: df.loc[v, 'sr'].mean() for k, v in stack_map.items()}
stack_sr = pd.Series(stack_sr).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
stack_sr.iloc[::-1].plot(kind='barh', ax=ax, color='teal')
ax.set_title('Средняя senior-зарплата по стеку'); ax.set_xlabel('USD/год')
ax.xaxis.set_major_formatter(lambda x, _: f'${x/1000:.0f}k')
plt.tight_layout(); plt.show()

zk = df.loc['ZK Engineer', 'sr']; sol = df.loc['Smart Contract Engineer', 'sr']
print('Рейтинг стеков (senior):')
print(stack_sr.round(0).to_string())
print(f'\nZK дороже Solidity (Smart Contract) на {(zk/sol-1)*100:.0f}%  (${zk-sol:,.0f})')

## Упражнение 2. Риск токенов (Монте-Карло)

Варьируем волатильность $\sigma$ и долю токенов. Считаем $P(C_{real}<0.5\,C)$ и
**риск-скорректированную** компенсацию $\;\mathbb{E}[C]-\text{std}[C]$.

In [ ]:
def simulate_comp(total, token_pct, sigma, n=200_000, seed=0):
    rng = np.random.default_rng(seed)
    base = (1 - token_pct) * total
    grant = token_pct * total
    m = rng.lognormal(mean=0.0, sigma=sigma, size=n)
    return base + m * grant

total = 250_000
sigmas = [0.4, 0.8, 1.2]
tokens = [0.15, 0.25, 0.35]

rows = []
for th in tokens:
    for sg in sigmas:
        c = simulate_comp(total, th, sg)
        grant = th * total
        # реализованная стоимость токен-гранта = c - base
        token_val = c - (1 - th) * total
        rows.append({'token_pct': th, 'sigma': sg,
                     'mean': c.mean(), 'std': c.std(),
                     'P(токен −50%)': (token_val < 0.5*grant).mean(),
                     'VaR5% (недобор)': total - np.percentile(c, 5),
                     'risk_adj (mean-std)': c.mean() - c.std()})
res = pd.DataFrame(rows)
print(res.to_string(index=False,
      formatters={'mean': '${:,.0f}'.format, 'std': '${:,.0f}'.format,
                  'risk_adj (mean-std)': '${:,.0f}'.format,
                  'VaR5% (недобор)': '${:,.0f}'.format,
                  'P(токен −50%)': '{:.1%}'.format}))

pivot = res.pivot(index='token_pct', columns='sigma', values='VaR5% (недобор)')
fig, ax = plt.subplots(figsize=(7, 4))
sns.heatmap(pivot, annot=True, fmt=',.0f', cmap='Reds', ax=ax)
ax.set_title('VaR 5%: недобор к номиналу ($), 5-й перцентиль')
plt.tight_layout(); plt.show()

print('\nВывод: P(токен теряет >50%) зависит в основном от волатильности σ,')
print('а долларовый недобор (VaR) растёт и с σ, и с долей токенов token_pct;')
print('риск-скорректированная компенсация (mean−std) при этом падает.')

## Упражнение 3. Контрактор vs штат

Крипто-компании часто нанимают **контракторами** (без соц. пакета, но выше ставка).
Сравним чистыми для дохода $150k: штатный сотрудник (US) vs self-employed
контрактор (платит обе части FICA — self-employment tax 15.3%).

In [ ]:
FED_2024 = [(0,0.10),(11600,0.12),(47150,0.22),(100525,0.24),(191950,0.32),
            (243725,0.35),(609350,0.37)]
STD_DED = 14600

def federal_tax(income):
    taxable = max(0.0, income - STD_DED)
    tax, prev = 0.0, None
    for i, (lo, rate) in enumerate(FED_2024):
        hi = FED_2024[i+1][0] if i+1 < len(FED_2024) else float('inf')
        if taxable > lo:
            tax += (min(taxable, hi) - lo) * rate
        else:
            break
    return tax

def net_employee(gross):
    # сотрудник: половина FICA (7.65%) + подоходный
    fica = gross * 0.0765
    return gross - fica - federal_tax(gross)

def net_contractor(gross, rate_premium=0.0):
    g = gross * (1 + rate_premium)
    # self-employment tax 15.3% на 92.35% дохода; половина SE-налога вычитается
    se_base = g * 0.9235
    se_tax = se_base * 0.153
    taxable_income = g - se_tax/2
    return g - se_tax - federal_tax(taxable_income)

base = 150_000
print(f'Штатный сотрудник ${base:,}: на руки ${net_employee(base):,.0f}')
for prem in [0.0, 0.10, 0.20]:
    n = net_contractor(base, prem)
    print(f'Контрактор ставка +{prem:.0%}: на руки ${n:,.0f}  '
          f'({"больше" if n>net_employee(base) else "меньше"} штата)')

# при какой надбавке контрактор сравнивается со штатом?
target = net_employee(base)
lo, hi = 0.0, 0.5
for _ in range(60):
    mid = (lo+hi)/2
    if net_contractor(base, mid) < target: lo = mid
    else: hi = mid
print(f'\nБезубыточная надбавка контрактора ≈ +{mid:.1%} к ставке штата.')

## Упражнение 4. Живые данные RemoteOK

Тянем публичный API, считаем число вакансий по крипто-тегам и сравниваем со
срезом. При отсутствии сети — оффлайн-фолбэк (код запроса остаётся рабочим).

In [ ]:
import json
from urllib.request import Request, urlopen
from collections import Counter

CRYPTO_TAGS = {'crypto','web3','blockchain','solidity','defi','ethereum',
               'bitcoin','nft','smart contract','rust'}

def fetch_remoteok(timeout=6):
    req = Request('https://remoteok.com/api',
                  headers={'User-Agent': 'Mozilla/5.0 (research notebook)'})
    with urlopen(req, timeout=timeout) as r:
        data = json.loads(r.read().decode('utf-8'))
    return [j for j in data if isinstance(j, dict) and j.get('position')]

try:
    jobs = fetch_remoteok()
    cnt = Counter()
    for j in jobs:
        tags = {str(t).lower() for t in j.get('tags', [])}
        for t in (tags & CRYPTO_TAGS):
            cnt[t] += 1
    if not cnt:
        raise ValueError('нет крипто-тегов')
    src = 'RemoteOK (live)'
except Exception as e:
    print(f'⚠️ Сеть недоступна ({type(e).__name__}). Оффлайн-срез тегов.')
    cnt = Counter({'web3': 42, 'solidity': 31, 'blockchain': 28, 'crypto': 25,
                   'rust': 18, 'defi': 15, 'ethereum': 12, 'smart contract': 9})
    src = 'offline snapshot'

tags_s = pd.Series(dict(cnt)).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(9, 4))
tags_s.plot(kind='bar', ax=ax, color='indigo')
ax.set_title(f'Число удалённых вакансий по крипто-тегам ({src})')
ax.set_ylabel('вакансий'); plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()
print(tags_s.to_string())

## Упражнение 5. Персональный рейтинг и его устойчивость

Задаём свои веса (деньги / спрос / стабильность) и проверяем, насколько выбор
топ-роли устойчив к возмущению весов на ±0.1.

In [ ]:
def zscore(s): return (s - s.mean())/s.std(ddof=0)
def composite(data, w):
    sc = pd.Series(0.0, index=data.index)
    for col, wv in w.items(): sc += wv*zscore(data[col])
    return sc

# личные веса: деньги важнее, но ценю стабильность
my_w = {'sr': 0.45, 'openings_pct': 0.25, 'token_pct': -0.30}
base_rank = composite(df, my_w).sort_values(ascending=False)
print('Мой топ-5:')
print(base_rank.head(5).round(3).to_string())
top1 = base_rank.index[0]

# устойчивость: 500 случайных возмущений весов на ±0.1
rng = np.random.default_rng(0)
wins = Counter()
for _ in range(500):
    w = {k: v + rng.uniform(-0.1, 0.1) for k, v in my_w.items()}
    wins[composite(df, w).idxmax()] += 1

stab = pd.Series(dict(wins)).sort_values(ascending=False)/500
print(f'\nБазовый выбор #1: {top1}')
print('Доля побед роли как #1 при возмущении весов ±0.1:')
print(stab.round(3).to_string())
print(f'\nУстойчивость выбора «{top1}»: {stab.get(top1, 0):.1%} прогонов.')

## Итоги

- **Стек:** ZK — самый дорогой стек (senior), далее Solidity/Rust; массовый
  спрос — у TS/React (Frontend/Full-Stack).
- **Токены:** риск глубокой просадки растёт и с долей токенов, и с волатильностью;
  риск-скорректированная компенсация при этом снижается.
- **Контрактор vs штат:** контрактор платит полный SE-налог 15.3%, поэтому нужна
  надбавка к ставке (≈безубыточный процент из упр. 3), чтобы обойти штат по чистыми.
- **Персональный выбор** стоит проверять на устойчивость к весам — если лидер
  меняется от малого сдвига, решение «хрупкое».